# 32

Combine multiple CSV files into a single DataFrame, extracting city and state from the filename:

In [30]:
import pandas as pd
import numpy as np
import glob
import os

In [31]:
all_dfs = []

for one_filename in glob.glob('./data/*,*.csv'):
    print(f'Loading {one_filename}...')

    # Use os.path.basename to get just the filename without the directory path
    basename = os.path.basename(one_filename)

    # Extract city and state from the basename
    city, state = basename.removesuffix('.csv').split(',')

    one_df = (
        pd
        .read_csv(one_filename,
                  usecols=[0, 1, 2],
                  names=['date_time',
                         'max_temp',
                         'min_temp'],
                  header=0)
        .assign(city=city.replace('+', ' ').title(),
                state=state.upper())
    )

    all_dfs.append(one_df)

df = pd.concat(all_dfs)

Loading ./data\albany,ny.csv...
Loading ./data\boston,ma.csv...
Loading ./data\chicago,il.csv...
Loading ./data\los+angeles,ca.csv...
Loading ./data\new+york,ny.csv...
Loading ./data\san+francisco,ca.csv...
Loading ./data\springfield,il.csv...
Loading ./data\springfield,ma.csv...


Does the data for each city and state start and end at (roughly) the same time?

In [35]:
df.groupby(['state', 'city'])['date_time'].min().sort_values()

state  city         
CA     Los Angeles      2018-12-11 00:00:00
       San Francisco    2018-12-11 00:00:00
IL     Chicago          2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
MA     Boston           2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
NY     Albany           2018-12-11 00:00:00
       New York         2018-12-11 00:00:00
Name: date_time, dtype: object

In [36]:
df.groupby(['state', 'city'])['date_time'].max().sort_values()

state  city         
CA     Los Angeles      2019-03-11 21:00:00
       San Francisco    2019-03-11 21:00:00
IL     Chicago          2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
MA     Boston           2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
NY     Albany           2019-03-11 21:00:00
       New York         2019-03-11 21:00:00
Name: date_time, dtype: object

In [37]:
df.groupby(['state', 'city'])['date_time'].agg(['min', 'max'])

min                  max
state city                                                   
CA    Los Angeles    2018-12-11 00:00:00  2019-03-11 21:00:00
      San Francisco  2018-12-11 00:00:00  2019-03-11 21:00:00
IL    Chicago        2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
MA    Boston         2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
NY    Albany         2018-12-11 00:00:00  2019-03-11 21:00:00
      New York       2018-12-11 00:00:00  2019-03-11 21:00:00

What is the lowest minimum temperature recorded for each city in the data set?

In [41]:
df.groupby(['state', 'city'])['min_temp'].min()

state  city         
CA     Los Angeles       4
       San Francisco     3
IL     Chicago         -28
       Springfield     -25
MA     Boston          -14
       Springfield     -20
NY     Albany          -19
       New York        -14
Name: min_temp, dtype: int64

What is the highest maximum temperature recorded in each state in the data set?

In [40]:
df.groupby('state')['max_temp'].max()

state
CA    23
IL    16
MA    17
NY    15
Name: max_temp, dtype: int64

Run "describe" on the minimum and maximum temperature for each state-city combination

In [38]:
df.groupby(['state', 'city'])[['max_temp', 'min_temp']].apply(pd.DataFrame.describe)

max_temp    min_temp
state city                                     
CA    Los Angeles count  728.000000  728.000000
                  mean    17.054945   10.637363
                  std      2.708640    2.705200
                  min     12.000000    4.000000
                  25%     15.000000    9.000000
...                             ...         ...
NY    New York    min    -12.000000  -14.000000
                  25%      2.000000   -4.000000
                  50%      4.000000    0.000000
                  75%      7.000000    2.000000
                  max     15.000000   12.000000

[64 rows x 2 columns]

What is the average difference in temperature (i.e., max - min) for each of the cities in our data set?

In [39]:
df.groupby(['state', 'city'])[['min_temp', 'max_temp']].apply(lambda g: np.mean(g.max() - g.min()) )

state  city         
CA     Los Angeles      12.0
       San Francisco     8.0
IL     Chicago          34.0
       Springfield      35.5
MA     Boston           26.0
       Springfield      28.5
NY     Albany           26.5
       New York         26.5
dtype: float64